# Notebook 02 — Data Cleaning

Cleans **Dataset 2** (`career_dataset_large.xlsx`) and produces
`data/processed/career_training.csv` with exactly four columns:
`Education Level`, `Specialization`, `Skills`, `Recommended Career`.

Normalization functions are imported from `utils/preprocessing.py`.

---
## 0. Path setup — add project root to `sys.path`

In [ ]:
import sys
import os

# Notebook lives in AI-Career-Recommendation/notebooks/
# Project root (containing utils/) is one level up.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print('Project root added to sys.path:', project_root)

---
## 1. Load Dataset 2

In [ ]:
import pandas as pd

ds2_path = '../data/raw/career_dataset_large.xlsx'
df = pd.read_excel(ds2_path, engine='openpyxl')

print('=== Dataset 2 loaded ===')
print(f'Shape: {df.shape}')
print()
print('Columns:', df.columns.tolist())
print()
print('Null counts per column:')
print(df.isnull().sum())

---
## 2. Drop unwanted columns (PII and extra columns if present)

In [ ]:
# Columns to drop if present
cols_to_drop = ['Certifications', 'CGPA/Percentage', 'Name', 'Age', 'CandidateID']
existing_to_drop = [c for c in cols_to_drop if c in df.columns]

if existing_to_drop:
    df = df.drop(columns=existing_to_drop)
    print(f'Dropped columns: {existing_to_drop}')
else:
    print('No columns to drop — dataset already contains only the required columns.')

print(f'Remaining columns: {df.columns.tolist()}')

---
## 3. Import normalization functions

In [ ]:
from utils.preprocessing import (
    normalize_education,
    normalize_specialization,
    normalize_skills_list,
    normalize_career,
    split_semicolon_field,
)

print('Normalization functions imported successfully.')

---
## 4. Detect Skills column delimiter

In [ ]:
# Detect delimiter: inspect first non-null value
first_skill_val = df['Skills'].dropna().iloc[0]
skill_delimiter = ';' if ';' in first_skill_val else ','
print(f'Detected skill delimiter: {repr(skill_delimiter)}')
print(f'Example raw Skills value: {repr(first_skill_val)}')

---
## 5. Apply normalization

In [ ]:
# 5a. Normalize Education Level
df['Education Level'] = df['Education Level'].astype(str).apply(normalize_education)
print('Education Level normalized. Value counts:')
print(df['Education Level'].value_counts().head(10))

In [ ]:
# 5b. Normalize Specialization
df['Specialization'] = df['Specialization'].astype(str).apply(normalize_specialization)
print('Specialization normalized. Top-10 values:')
print(df['Specialization'].value_counts().head(10))

In [ ]:
# 5c. Split and normalize Skills
#
# split_semicolon_field handles ';'-delimited data.
# For ',' delimiter we split manually, then pass the list through normalize_skills_list.
# Result is stored as a ';'-joined string for CSV compatibility.

def split_skills(raw: str) -> list:
    """Split raw skills string using detected delimiter."""
    if skill_delimiter == ';':
        return split_semicolon_field(raw)
    else:  # comma delimiter
        return [token.strip() for token in raw.split(',') if token.strip()]

df['Skills'] = (
    df['Skills']
    .astype(str)
    .apply(lambda raw: ';'.join(normalize_skills_list(split_skills(raw))))
)

print('Skills normalized. Sample values:')
print(df['Skills'].head(5).tolist())

In [ ]:
# 5d. Normalize Recommended Career
df['Recommended Career'] = df['Recommended Career'].astype(str).apply(normalize_career)
print('Recommended Career normalized. Value counts:')
print(df['Recommended Career'].value_counts())

---
## 6. Keep only the four target columns, drop nulls and duplicates

In [ ]:
keep_cols = ['Education Level', 'Specialization', 'Skills', 'Recommended Career']
df = df[keep_cols].copy()

rows_start = len(df)

# Replace empty strings with NaN so dropna catches them
df.replace('', float('nan'), inplace=True)

# Drop rows where any required column is null/empty
df.dropna(subset=keep_cols, inplace=True)
print(f'After null-drop:  {len(df)} rows  (removed {rows_start - len(df)})')

# Drop fully duplicate rows
rows_before_dedup = len(df)
df.drop_duplicates(inplace=True)
print(f'After dedup:      {len(df)} rows  (removed {rows_before_dedup - len(df)})')

---
## 7. Save to `data/processed/career_training.csv`

In [ ]:
output_path = '../data/processed/career_training.csv'
df.to_csv(output_path, index=False)

print(f'Saved {len(df)} rows to: {output_path}')
print(f'Columns: {df.columns.tolist()}')

---
## 8. Validation

In [ ]:
# Reload from disk to validate the saved file
df_check = pd.read_csv(output_path)

print('=== Validation ===')
print(f'Row count: {len(df_check)}')

In [ ]:
# 8.1 Assert row count > 4000
assert len(df_check) > 4000, f'Expected > 4000 rows, got {len(df_check)}'
print(f'[PASS] Row count {len(df_check)} > 4000')

In [ ]:
# 8.2 Career class distribution — confirm no class has 0 rows
career_dist = df_check['Recommended Career'].value_counts()
print('Career class distribution:')
print(career_dist.to_string())
print()
assert (career_dist > 0).all(), 'Some career class has 0 rows!'
print(f'[PASS] All {len(career_dist)} career classes have at least 1 row.')

In [ ]:
# 8.3 Assert no null values in any column
null_counts = df_check.isnull().sum()
print('Null counts per column:')
print(null_counts)
assert null_counts.sum() == 0, f'Found null values: {null_counts[null_counts > 0].to_dict()}'
print('[PASS] No null values in any column.')

In [ ]:
# 8.4 Assert no fully duplicate rows
dup_count = df_check.duplicated().sum()
print(f'Duplicate row count: {dup_count}')
assert dup_count == 0, f'Found {dup_count} duplicate rows!'
print('[PASS] No fully duplicate rows.')

In [ ]:
print()
print('=== All validations passed ===')
print(f'career_training.csv: {len(df_check)} rows, columns: {df_check.columns.tolist()}')